In [1]:
# bebuhk
# 04.08.2026: copied from /Users/bene/Code/V_ext_jax/___tail_correction_test.ipynb. simplified and free from dependencies.

# Understanding tail correction in RASPA (and cDFT)

**Author:** Bene  
**Created:** 2026-08-02 

**Description:** Discomposing the RASPA tail correction energy into $U_{tail,s-s} + U_{tail,s-f} + U_{tail,f-f}$. Only the part of the solid fluid interactions $U_{tail,s-f}$ is relevant for cDFT.

In [2]:
import numpy as np

## Tail Correction of CO2 in RSM0016

The tail correction of CO2 in RASPA 2.0 for a 111 supercell (i.e. a single unitcell) is $$U_{tail,RASPA} = -147.67313013 K$$ (i.e. for RSM0016 with Vincents UFF parameters and a signle TraPPE CO2 inserted).

Lets recompute the individual parts of it:

$$U_{tail,RASPA} = U_{tail,s-s} + U_{tail,s-f} + U_{tail,f-f},$$

i.e. a contribution for the solid-solid, the solid-fluid, and the fluid-fluid interactions. For cDFT, only the part of the tail correction regarding the solid-fluid interactions $U_{tail,s-f}$ is relevant.



Tail correction is defined as 

$$U_{tail,RASPA} = N\cdot u_{tail,RASPA}$$

$$u_{\text{tail}} = 2\pi\rho\int_{r_c}^{\infty} r^2\,u(r)\,\mathrm{d}r
= \frac{8}{3}\pi\rho\,\varepsilon\sigma^{3}
\left[\frac{1}{3}\left(\frac{\sigma}{r_c}\right)^{9}-\left(\frac{\sigma}{r_c}\right)^{3}\right]$$

$$\rho = N / V$$

#### LJ Parameters

In [3]:
sigmas_co2 = [3.05, 2.8, 3.05]
epsilons_co2 = [79.0, 27.0, 79.0]

In [4]:
sigmas_atoms_RSM0016 = np.array([2.8045, 2.8045, 2.5711, 2.5711, 2.5711, 2.5711, 2.5711, 2.5711,
   2.5711, 2.5711, 2.5711, 2.5711, 2.5711, 2.5711, 2.5711, 2.5711,
   2.5711, 2.5711, 3.4309, 3.4309, 3.4309, 3.4309, 3.2607, 3.2607,
   3.2607, 3.2607])
epsilons_atoms_RSM0016 = np.array([18.1178, 18.1178, 22.1439, 22.1439, 22.1439, 22.1439, 22.1439,
   22.1439, 22.1439, 22.1439, 22.1439, 22.1439, 22.1439, 22.1439,
   22.1439, 22.1439, 22.1439, 22.1439, 52.8435, 52.8435, 52.8435,
   52.8435, 34.7257, 34.7257, 34.7257, 34.7257])
lattice_RSM0016 = np.array([[ 4.0168    ,  0.        ,  0.        ],
   [-1.97880151,  9.35914831,  0.        ],
   [-1.97956238, -0.03108988,  9.34428816]])
volume_cell_RSM0016 = np.abs(
    np.dot(lattice_RSM0016[0], np.cross(lattice_RSM0016[1], lattice_RSM0016[2]))
)  # a1 dot (a2 x a3) [Å^3]

###
Separating the contribution from a single interaction $$\boxed{I(\sigma, \varepsilon) = \frac{8}{3}\pi\, \frac{1}{V} \,\varepsilon\sigma^{3}
\left[\frac{1}{3}\left(\frac{\sigma}{r_c}\right)^{9}-\left(\frac{\sigma}{r_c}\right)^{3}\right]\,} \tag{1}$$ lets us write

$$U_{tail,RASPA} = N^2 \cdot \frac{8}{3}\pi\, \frac{1}{V} \,\varepsilon\sigma^{3}
\left[\frac{1}{3}\left(\frac{\sigma}{r_c}\right)^{9}-\left(\frac{\sigma}{r_c}\right)^{3}\right] =  N^2 \cdot I(\sigma, \varepsilon)$$

which is how RASPA computes the tail correction for an N particle system with equal LJ parameters $\sigma$ and $\varepsilon$. 

In [5]:
def tail_single_interaction(epsilon_ij, sigma_ij, rcut=14.0, volume_cell=volume_cell_RSM0016):
    return (
        2* np.pi / volume_cell* 4/ 3
        * epsilon_ij* sigma_ij**3
        * (1 / 3 * (sigma_ij / rcut) ** 9 - (sigma_ij / rcut) ** 3)
    )
#tail_single_interaction(epsilon_ij=10.0, sigma_ij=1.0)

### Total tail correction $U_{tail,RASPA}$

For particles with their individual LJ parameters, the tail correction is computed as

$$\boxed{U_{tail,RASPA} = \sum_i^N \sum_j^N \cdot I(\sigma_{ij}, \varepsilon_{ij}) \,} \tag{2}$$

where the pair LJ parameters are computed via the Lorentz-Berthelot combining rules, i.e. $\sigma_{ij} = \frac{\sigma_{i} + \sigma_{j}}{2}$ and $\varepsilon_{ij} = \sqrt{\varepsilon_{i} \varepsilon_{j}}$.

In [6]:
all_sigs = np.append(sigmas_atoms_RSM0016, sigmas_co2)
all_sigs

array([2.8045, 2.8045, 2.5711, 2.5711, 2.5711, 2.5711, 2.5711, 2.5711,
       2.5711, 2.5711, 2.5711, 2.5711, 2.5711, 2.5711, 2.5711, 2.5711,
       2.5711, 2.5711, 3.4309, 3.4309, 3.4309, 3.4309, 3.2607, 3.2607,
       3.2607, 3.2607, 3.05  , 2.8   , 3.05  ])

In [7]:
all_eps = np.append(epsilons_atoms_RSM0016, epsilons_co2)
all_eps

array([18.1178, 18.1178, 22.1439, 22.1439, 22.1439, 22.1439, 22.1439,
       22.1439, 22.1439, 22.1439, 22.1439, 22.1439, 22.1439, 22.1439,
       22.1439, 22.1439, 22.1439, 22.1439, 52.8435, 52.8435, 52.8435,
       52.8435, 34.7257, 34.7257, 34.7257, 34.7257, 79.    , 27.    ,
       79.    ])

In [8]:
def compute_U_tail_for_set(set_epsilons, set_sigmas, rcut=14.0, volume_cell=volume_cell_RSM0016):
    assert len(set_epsilons) == len(set_sigmas), "set_epsilons and set_sigmas must have the same length."
    U_tail = 0
    for i in range(len(set_sigmas)):
        for j in range(len(set_sigmas)):
            # if i == j:  #no, also self interactions are included
            #     continue
            U_tail += tail_single_interaction(epsilon_ij=np.sqrt(set_epsilons[i] * set_epsilons[j]), sigma_ij=(set_sigmas[i] + set_sigmas[j]) / 2)
    return U_tail
U_tail_RASPA = compute_U_tail_for_set(set_epsilons=all_eps, set_sigmas=all_sigs)
print(f"Total tail correction energy:\t{U_tail_RASPA} K per molecule.\n(RASPA gives\t\t\t-147.67313013 K per molecule.)")


Total tail correction energy:	-147.67313016866518 K per molecule.
(RASPA gives			-147.67313013 K per molecule.)


### Tail correction by contributions $U_{tail,RASPA} = U_{tail,s-s} + U_{tail,s-f} + U_{tail,f-f},$

Splitting the contributions of the tail corrections into s-s, s-f, and f-f becomes possible by destiguishing between solid and fluid particles as $N = N_s + N_f$:
$$U_{tail,RASPA} = \sum_i^{N}\sum_j^{N} I(\sigma_{ij},\varepsilon_{ij})
= \underbrace{\sum_i^{N_s}\sum_j^{N_s} I(\sigma_{ij},\varepsilon_{ij})}_{U_{\text{tail},\,s\text{-}s}}
+ \underbrace{\sum_i^{N_s}\sum_j^{N_f} I(\sigma_{ij},\varepsilon_{ij})
            + \sum_i^{N_f}\sum_j^{N_s} I(\sigma_{ij},\varepsilon_{ij})}_{U_{\text{tail},\,s\text{-}f}}
+ \underbrace{\sum_i^{N_f}\sum_j^{N_f} I(\sigma_{ij},\varepsilon_{ij})}_{U_{\text{tail},\,f\text{-}f}}$$

In [9]:
U_tail_ss = compute_U_tail_for_set(set_epsilons=epsilons_atoms_RSM0016, set_sigmas=sigmas_atoms_RSM0016)
U_tail_ff = compute_U_tail_for_set(set_epsilons=epsilons_co2, set_sigmas=sigmas_co2)
U_tail_sf = 0
for i in range(len(sigmas_atoms_RSM0016)):
    for j in range(len(sigmas_co2)):
        # if i == j:  #no, also self interactions are included
        #     continue
        U_tail_sf += tail_single_interaction(epsilon_ij=np.sqrt(epsilons_atoms_RSM0016[i] * epsilons_co2[j]), sigma_ij=(sigmas_atoms_RSM0016[i] + sigmas_co2[j]) / 2)
U_tail_sf *= 2
U_tail_RASPA = U_tail_ss + U_tail_sf + U_tail_ff
print(f"Total tail correction energy:\t{U_tail_RASPA} K per molecule.\n(RASPA gives\t\t\t-147.67313013 K per molecule.)\n")
print(f"end=end check: U_tail_ss + U_tail_sf + U_tail_ff = {U_tail_ss} + {U_tail_sf} + {U_tail_ff} = {U_tail_ss + U_tail_sf + U_tail_ff}")

Total tail correction energy:	-147.67313016866535 K per molecule.
(RASPA gives			-147.67313013 K per molecule.)

end=end check: U_tail_ss + U_tail_sf + U_tail_ff = -106.65947581126268 + -37.68434051982547 + -3.329313837577209 = -147.67313016866535


Thus, the cDFT-relevant tail corretion, i.e. $U_{tail,s-f}$, equals -37.68434051982547 K.

## Rest (some earlier verifications)

### RSM0016 and CH4

In [10]:
if "sigma_segment_i" not in locals():
    sigma_segment_i = 3.73
    epsilon_segment_i = 148.0
    print("no segment LJ parameters loaded yet. unsing CH4 TraPPE LJ parameters.")

sigma_segment_i, epsilon_segment_i

no segment LJ parameters loaded yet. unsing CH4 TraPPE LJ parameters.


(3.73, 148.0)

In [11]:
if "sigmas_atoms" not in locals():
    sigmas_atoms = np.array([2.8045, 2.8045, 2.5711, 2.5711, 2.5711, 2.5711, 2.5711, 2.5711,
       2.5711, 2.5711, 2.5711, 2.5711, 2.5711, 2.5711, 2.5711, 2.5711,
       2.5711, 2.5711, 3.4309, 3.4309, 3.4309, 3.4309, 3.2607, 3.2607,
       3.2607, 3.2607])
    epsilons_atoms = np.array([18.1178, 18.1178, 22.1439, 22.1439, 22.1439, 22.1439, 22.1439,
       22.1439, 22.1439, 22.1439, 22.1439, 22.1439, 22.1439, 22.1439,
       22.1439, 22.1439, 22.1439, 22.1439, 52.8435, 52.8435, 52.8435,
       52.8435, 34.7257, 34.7257, 34.7257, 34.7257])
    lattice = np.array([[ 4.0168    ,  0.        ,  0.        ],
       [-1.97880151,  9.35914831,  0.        ],
       [-1.97956238, -0.03108988,  9.34428816]])
    print("No sigmas loaded yet. setting LJ parameters to atoms in RSM0016.cif.")

No sigmas loaded yet. setting LJ parameters to atoms in RSM0016.cif.


In [12]:
include_self_interactions = False#True

epsilons_mix_LJ = np.sqrt(epsilons_atoms * epsilon_segment_i)
sigmas_mix_LJ = (sigmas_atoms + sigma_segment_i) / 2


# keeping each one twice. (essentially same as doubling sum in the end)
epsilons_mix_LJ = np.append(epsilons_mix_LJ, epsilons_mix_LJ)
sigmas_mix_LJ = np.append(sigmas_mix_LJ, sigmas_mix_LJ)


if include_self_interactions:
    epsilons_mix_LJ = np.append(epsilons_mix_LJ, epsilon_segment_i)
    sigmas_mix_LJ = np.append(sigmas_mix_LJ, sigma_segment_i)

rcut = 14
volume_cell = np.abs(
    np.dot(lattice[0], np.cross(lattice[1], lattice[2]))
)  # a1 dot (a2 x a3) [Å^3]
tail_corrections = (
    2
    * np.pi
    / volume_cell
    * 4
    / 3
    * epsilons_mix_LJ
    * sigmas_mix_LJ**3
    * (1 / 3 * (sigmas_mix_LJ / rcut) ** 9 - (sigmas_mix_LJ / rcut) ** 3)
)  # array with tail correction due to each atom. [K / molecule]
sum(tail_corrections)  # total tail correction for the molecule. [K / molecule]

-39.801689626751575

### RSM0016 with co2

In [13]:
if "sigma_segment_is" not in locals():
    sigma_segment_is = [2.8, 3.05]
    epsilon_segment_is = [27.0, 79.0]
    print("no segment LJ parameters loaded yet. unsing CO2 TraPPE LJ parameters.")

sigma_segment_is, epsilon_segment_is

no segment LJ parameters loaded yet. unsing CO2 TraPPE LJ parameters.


([2.8, 3.05], [27.0, 79.0])

In [14]:
if "sigmas_atoms" not in locals():
    sigmas_atoms = np.array([2.8045, 2.8045, 2.5711, 2.5711, 2.5711, 2.5711, 2.5711, 2.5711,
       2.5711, 2.5711, 2.5711, 2.5711, 2.5711, 2.5711, 2.5711, 2.5711,
       2.5711, 2.5711, 3.4309, 3.4309, 3.4309, 3.4309, 3.2607, 3.2607,
       3.2607, 3.2607])
    epsilons_atoms = np.array([18.1178, 18.1178, 22.1439, 22.1439, 22.1439, 22.1439, 22.1439,
       22.1439, 22.1439, 22.1439, 22.1439, 22.1439, 22.1439, 22.1439,
       22.1439, 22.1439, 22.1439, 22.1439, 52.8435, 52.8435, 52.8435,
       52.8435, 34.7257, 34.7257, 34.7257, 34.7257])
    lattice = np.array([[ 4.0168    ,  0.        ,  0.        ],
       [-1.97880151,  9.35914831,  0.        ],
       [-1.97956238, -0.03108988,  9.34428816]])
    print("No sigmas loaded yet. setting LJ parameters to atoms in RSM0016.cif.")

In [15]:
include_self_interactions = False#True

epsilons_mix_LJ = np.sqrt(epsilons_atoms * epsilon_segment_is[1])
sigmas_mix_LJ = (sigmas_atoms + sigma_segment_is[1]) / 2


# keeping each one twice. (essentially same as doubling sum in the end)
epsilons_mix_LJ = np.append(epsilons_mix_LJ, epsilons_mix_LJ)
sigmas_mix_LJ = np.append(sigmas_mix_LJ, sigmas_mix_LJ)


if include_self_interactions:
    epsilons_mix_LJ = np.append(epsilons_mix_LJ, epsilon_segment_i)
    sigmas_mix_LJ = np.append(sigmas_mix_LJ, sigma_segment_i)

rcut = 14
volume_cell = np.abs(
    np.dot(lattice[0], np.cross(lattice[1], lattice[2]))
)  # a1 dot (a2 x a3) [Å^3]
tail_corrections = (
    2
    * np.pi
    / volume_cell
    * 4
    / 3
    * epsilons_mix_LJ
    * sigmas_mix_LJ**3
    * (1 / 3 * (sigmas_mix_LJ / rcut) ** 9 - (sigmas_mix_LJ / rcut) ** 3)
)  # array with tail correction due to each atom. [K / molecule]
sum(tail_corrections)  # total tail correction for the molecule. [K / molecule]

-15.355539387419066

In [16]:
-6.973261744987322 # C_co2 tail correction 
-15.355539387419066 # O_co2 tail correction

-15.355539387419066

In [17]:
-15.355539387419066 * 2 + -6.973261744987322 * 1 # = -37.684340519825454
# HEURICA! (vgl raspa: -37,6843405104)

-37.684340519825454